# 01 · API exploration

Phase 1 of the project: call each NHL endpoint, look at the raw JSON, and note the nested structures, keys and quirks the parsers have to handle.

| Table | Endpoint |
|---|---|
| teams, standings | `/v1/standings/now` |
| players | `/v1/roster/{team}/{season or current}` |
| games | `/v1/club-schedule-season/{team}/{season}` |
| game_stats | provided dataset, or `/v1/gamecenter/{game_id}/boxscore` |
| skater / goalie season stats | provided datasets, or `/v1/club-stats/{team}/{season}/2` |

The client waits `NHL_REQUEST_DELAY` seconds between calls, retries on 429/5xx, and writes every raw response to `data/raw/api_cache/`.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nhl_pipeline import config, parsers
from nhl_pipeline.api_client import NHLClient
from nhl_pipeline.demo import DemoClient

# "api"  = live NHL public API (no key needed)
# "demo" = offline synthetic season with the same JSON shape (no network)
SOURCE = "api"

client = DemoClient() if SOURCE == "demo" else NHLClient()
pd.set_option("display.max_columns", 40)
print("Source:", SOURCE, "| season:", config.SEASON, "| base URL:", config.BASE_URL)

## Helper: summarise a JSON payload
Prints the shape of nested objects without dumping thousands of lines.

In [ ]:
def shape(obj, depth=0, max_depth=3, key="root"):
    pad = "  " * depth
    if isinstance(obj, dict):
        print(f"{pad}{key}: dict[{len(obj)}]")
        if depth < max_depth:
            for k, v in list(obj.items())[:25]:
                shape(v, depth + 1, max_depth, k)
    elif isinstance(obj, list):
        print(f"{pad}{key}: list[{len(obj)}]")
        if obj and depth < max_depth:
            shape(obj[0], depth + 1, max_depth, "[0]")
    else:
        print(f"{pad}{key}: {type(obj).__name__} = {str(obj)[:40]!r}")

## 1. Standings → `teams` + `standings`

In [ ]:
standings_json = client.standings()
shape(standings_json, max_depth=2)

In [ ]:
first = standings_json["standings"][0]
{k: first.get(k) for k in ["teamAbbrev", "teamName", "conferenceName", "divisionName", "teamLogo",
                           "gamesPlayed", "wins", "losses", "otLosses", "points", "goalFor",
                           "goalAgainst", "homeWins", "roadWins", "streakCode", "streakCount", "seasonId"]}

**Observations**
- Names are *localised objects* (`{"default": "Toronto", "fr": ...}`), not strings → `parsers.loc()` unwraps them.
- There is **no numeric team id** here, only `teamAbbrev`, so `teams.team_id` is generated on load.
- Goals fields are singular in the API (`goalFor`, `goalAgainst`).

In [ ]:
pd.json_normalize(standings_json["standings"])[
    ["teamAbbrev.default", "conferenceName", "divisionName", "gamesPlayed", "points"]].head()

## 2. Roster → `players`

In [ ]:
teams = sorted(parsers.loc(t["teamAbbrev"]) for t in standings_json["standings"])
print(len(teams), "teams:", teams)
roster_json = client.roster(teams[0])
print({group: len(players) for group, players in roster_json.items() if isinstance(players, list)})
roster_json["forwards"][0]

**Observations**
- Players come in **three lists** (`forwards`, `defensemen`, `goalies`) that must be flattened; `positionCode` distinguishes them.
- Height/weight are given in both imperial and metric; some records only carry one, so the parser converts when needed.
- `id` is the stable NHL player id → used as-is for `players.player_id`.

## 3. Club schedule → `games`

In [ ]:
schedule_json = client.schedule(teams[0])
games = pd.json_normalize(schedule_json["games"])
print(len(games), "games for", teams[0])
print(games["gameState"].value_counts().to_dict(), "| game types:", games["gameType"].value_counts().to_dict())
games.filter(regex="^(id|gameDate|gameType|gameState|homeTeam.abbrev|homeTeam.score|awayTeam.abbrev|awayTeam.score|venue.default)$").head()

**Observations**
- Future games have **no `score` key** → scores stay NULL.
- `gameState` codes: `OFF`/`FINAL` = final, `FUT` = upcoming, `LIVE`/`CRIT` = in progress. The DB stores raw codes; the dashboard maps labels.
- Each game appears in **both** clubs' schedules → dedupe on `game_id` (INSERT OR IGNORE).
- Preseason can include non-NHL opponents → those rows would break the FOREIGN KEY and are reported as orphans.

In [ ]:
valid, orphans = parsers.parse_schedule(schedule_json, {t: i for i, t in enumerate(teams, 1)}, config.SEASON)
print(f"{len(valid)} valid rows, {len(orphans)} orphan rows")

## 4. Box score → `game_stats` (fallback when the dataset file is absent)

In [ ]:
final_ids = games.loc[games["gameState"].isin(["OFF", "FINAL"]) & (games["gameType"] == 2), "id"]
if len(final_ids):
    box = client.boxscore(int(final_ids.iloc[0]))
    shape(box["playerByGameStats"], max_depth=3)
    display(pd.DataFrame(box["playerByGameStats"]["homeTeam"]["forwards"]).head())
else:
    print("No completed regular-season games yet for", teams[0])

**Observations**
- Skaters are split into `forwards` and `defense`; goalies are separate and excluded from `game_stats`.
- `toi` is `"MM:SS"` text, `sog` = shots on goal, `pim` = penalty minutes.

## 5. Club stats → `skater_season_stats` / `goalie_season_stats` (fallback)

In [ ]:
club = client.club_stats(teams[0])
print({k: (len(v) if isinstance(v, list) else v) for k, v in club.items()})
display(pd.DataFrame(club["skaters"]).head(3))
pd.DataFrame(club["goalies"]).head(3)

**Observations**
- Routing rule from the brief: `positionCode == "G"` → goalie table, anything else → skater table.
- Average TOI arrives in **seconds** (e.g. `1234.5`) → converted to `MM:SS`.
- A player traded mid-season appears under each club → unique key is `(player_id, season, team_id)`.

## 6. Provided datasets
Put the three files from the brief in `data/raw/datasets/` (any of CSV/JSON/XLSX). Notebook 02 uses them for tables 5–7 when present.

In [ ]:
from nhl_pipeline import datasets
for stem in ["game_stats", "skater_season_stats", "goalie_season_stats"]:
    f = datasets.find_dataset(stem)
    print(f"{stem:22}", f.name if f else "not found → will be built from the API")
    if f:
        display(datasets.read_any(f).head(3))

In [ ]:
print("Live API calls:", client.live_calls, "| cache hits:", client.cache_hits)